# 5. 임베딩과 의미 기반 분석
### 단어가 달라도 뜻이 비슷하면 찾아낸다

**환경:** Google Colab / Jupyter Notebook (인터넷 연결 필요, **GPU 런타임 권장**)
**준비물:** `TS_4-3` 엑셀 파일 (카메라/게임기/태블릿 상품평 6,001건)
**소요 시간:** 약 30분

## 오늘의 질문
단원 3·4의 **TF-IDF**는 **같은 단어**가 있어야 비슷하다고 판단합니다.

| 문장 1 | 문장 2 | TF-IDF의 판단 | 사람의 판단 |
|---|---|---|---|
| 배송이 느려요 | 택배가 늦게 왔어요 | 겹치는 단어 없음 → **다름** | **비슷함** |

> **임베딩**은 단어나 문장을 "뜻이 담긴 숫자 목록(벡터)"으로 바꿔서, 단어가 달라도 뜻이 비슷하면 가깝게 놓습니다.

## 실습 진행 과정
| 단계 | 할 일 | 확인할 결과 |
|---|---|---|
| 0. 준비 | 설치, 엑셀 읽기, 단어 나누기 | 리뷰 수 |
| ① 단어 임베딩 | Word2Vec 학습 → 가까운 단어 찾기 | 비슷한 단어 목록 |
| ② 문장 임베딩 | 사전학습 모델 **점검** → 문장 쌍 비교 (TF-IDF와 나란히) | 점검 결과, 문장 쌍 유사도 표 |
| ③ 임베딩 검색 | 리뷰를 **문장 단위**로 나눠 색인 → 질문과 뜻이 비슷한 리뷰 5개 찾기 | 두 방법의 검색 결과 |
| 정리 | 언제 무엇을 쓸까, 한계 | 비교 표 |

> **실행 방법:** 셀을 클릭하고 `Shift + Enter`. 반드시 **위에서부터 순서대로** 실행하세요.
> ✏️ 표시가 있는 곳은 **직접 바꿔 볼 수 있는 부분**입니다.
> ⚡ Colab 메뉴 **런타임 → 런타임 유형 변경 → GPU**를 고르면 ③이 훨씬 빨라집니다.

---
## 0. 준비
설치할 도구가 4개입니다. 처음 한 번만 실행하면 됩니다. (1~2분 걸릴 수 있음)

| 도구 | 하는 일 |
|---|---|
| `kiwipiepy` | 한국어 형태소 분석 (단어 나누기, 문장 나누기) |
| `openpyxl` | 엑셀 파일 읽기 |
| `gensim` | **Word2Vec** 단어 임베딩 학습 |
| `sentence-transformers` | **사전학습 문장 임베딩** 모델 사용 |

In [1]:
# 필요한 도구 설치 (처음 한 번만)
!pip install -q kiwipiepy openpyxl gensim sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 12.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 44.4 MB/s eta 0:00:00


Colab 왼쪽 **📁 파일 창**에 `TS_4-3`으로 시작하는 엑셀 파일을 끌어다 놓으세요.
아래 셀이 파일 이름을 **자동으로 찾습니다.**

In [2]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 엑셀 파일에서 '카테고리', '상품평' 두 열을 표(DataFrame)로 읽기
# ------------------------------------------------------------
# 표 다루기 도구와 파일 찾기 도구
import pandas as pd
import glob

# ===== ✏️ [입력] 찾을 엑셀 파일 이름 패턴 (*는 '아무 글자나') =====
# glob.glob(패턴): 현재 폴더에서 패턴에 맞는 파일 경로 리스트, [0]: 그 첫 번째 (없으면 IndexError)
excel_path = glob.glob("TS_4-3*.xlsx")[0]
# =========================================================================

# 엑셀에서 상품평 열만 읽고, 빈 리뷰는 빼기
df = pd.read_excel(excel_path, usecols=["상품평"]).dropna()
df["상품평"] = df["상품평"].astype(str)
print("파일:", excel_path)
print("리뷰 수:", len(df))

파일: TS_4-3.카메라게임기태블릿.xlsx
리뷰 수: 6001


> ⚠️ `IndexError: list index out of range`가 나면 파일을 아직 올리지 않았거나 이름이 `TS_4-3`으로 시작하지 않는 것입니다.

### 단어 나누기
Word2Vec과 TF-IDF는 **단어 목록**이 필요합니다. (10초 정도 걸림)

In [3]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 리뷰마다 형태소 분석을 하고, 필요한 단어만 골라
#                  '단어 리스트'를 담은 '단어' 열을 만듦
#                  (Word2Vec과 TF-IDF가 이 리스트를 입력으로 사용)
# ------------------------------------------------------------
# Kiwi 형태소 분석기
from kiwipiepy import Kiwi
kiwi = Kiwi()

# 형태소 분석 결과를 단어 목록으로 바꾸는 함수
def to_words(tokens):
    words = []
    for t in tokens:
        tag = t.tag                 # 'VA' 같은   태그
        if tag in {"VV", "VA"} or t.form == "않":
            words.append(t.form + "다")           # 동사·형용사·'않'은 '다' 붙이기
        elif tag == "XR":
            words.append(t.form + "하다")         # 어근은 '하다' 붙이기
        elif tag in {"NNG", "NNP"} or t.form in {"안", "못", "별로"}:
            words.append(t.form)                 # 명사와 부정·평가 표현은 그대로
    return words

# 리뷰 전체를 한 번에 분석해서 '단어' 열 만들기
# df["상품평"].tolist()      : '상품평' 열을 문자열 리스트로 변환
# kiwi.tokenize(문자열 목록)  : 리뷰마다 분석한 토큰 목록을 입력 순서대로 하나씩 돌려줌
# to_words(tokens)           : 리뷰 하나의 토큰 목록에서 남길 단어 리스트를 만듦
# [ ... for tokens in ... ]  : 모든 리뷰의 단어 리스트를 모은 리스트
# df["단어"] = 리스트         : '단어' 열로 저장 (리스트 순서대로 행에 들어감, 각 칸에는 문자열이 아니라 리스트가 들어감)
df["단어"] = [to_words(tokens) for tokens in kiwi.tokenize(df["상품평"].tolist())]

# 원문과 단어 목록 비교
print(df["상품평"][1])
print(df["단어"][1])

기존에 포토프린터를 가지고 있는데 그건 크고 전원을 연결해야 해서 손쉽게 들고 다닐 수는 없었어요. 그래서 이번에 초등 딸아이 크리스마스 선물로 카메라를 구입했어요. 크기도 작고 가벼워서 좋고 즉석 사진도 찍고 휴대폰에서 꾸며서 프린트도 할 수 있어서 아이가 좋아하네요. 인화하는 것보단 화질이 조금 떨어지는 것 같지만 즉석카메라라서 어쩔 수 없는 것 같아요. 우리 아이는 다이어리 꾸미기 하겠다면 신났습니다.
['기존', '포토', '프린터', '가지다', '크다', '전원', '연결', '들다', '다니다', '없다', '이번', '초등', '딸아이', '크리스마스', '선물', '카메라', '구입', '크기', '작다', '좋다', '즉석', '사진', '찍다', '휴대폰', '꾸미다', '프린트', '하다', '있다', '아이', '좋아하다', '인화', '화질', '떨어지다', '같다', '즉석', '카메라', '어쩌다', '없다', '같다', '아이', '다이어리', '꾸미다', '하다', '신나다']


---
## ① 단어 임베딩: Word2Vec

### 💡 어떻게 뜻을 배울까? — "친구를 보면 그 사람을 안다"
> "**배송**이 빨라요", "**택배**가 빨라요", "**배송**이 늦어요", "**택배**가 늦어요"

`배송`과 `택배`는 **주변에 같은 단어**(빠르다, 늦다)가 자주 나옵니다. Word2Vec은 이런 단어들을 **가까운 숫자 목록**으로 만듭니다.

| 설정 | 값 | 뜻 |
|---|---|---|
| `vector_size` | 100 | 단어 하나를 숫자 100개로 표현 |
| `window` | 5 | 앞뒤 5단어를 "주변 단어"로 봄 |
| `min_count` | 5 | 5번 미만 나온 단어는 학습하지 않음 |
| `seed`, `workers=1` | 42, 1 | 매번 같은 결과가 나오도록 고정 |

> 💡 Word2Vec은 데이터가 많을수록 좋아서 **리뷰 전체**로 학습합니다.

In [4]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 전체 리뷰의 단어 리스트로 Word2Vec 단어 임베딩 모델을 학습
# ------------------------------------------------------------
# Word2Vec 불러오기
from gensim.models import Word2Vec

# 리뷰 전체의 단어 목록으로 Word2Vec 학습 (10초 정도 걸림)
# Word2Vec(문장 목록, 설정들...) : 객체를 만들면서 바로 학습까지 수행
#   df["단어"].tolist()
#     - 학습 데이터, '단어 리스트들의 리스트' 형태
#     - 예: [['전', '쓰다', '좋다', ...], ['배송', '빠르다', ...], ...]
#   vector_size=100 : 단어 하나를 숫자 100개짜리 벡터로 표현
#   window=5        : 가운데 단어 기준 앞뒤 최대 5개 단어를 주변 단어(문맥)로 사용
#   min_count=5     : 전체에서 5번 미만 나온 단어는 단어 목록에서 제외하고 학습하지 않음
#   sg=1            : Skip-gram 방식으로 학습 (가운데 단어로 주변 단어를 예측), sg=0이면 CBOW 방식
#   epochs=10       : 전체 데이터를 10번 반복해서 학습
#   seed=42         : 벡터 초기값 등을 만드는 난수의 시작값을 고정
#   workers=1       : 학습 스레드를 1개만 사용
w2v = Word2Vec(df["단어"].tolist(), vector_size=100, window=5, min_count=5, sg=1, epochs=10, seed=42, workers=1)


# w2v.wv : 학습된 단어 벡터들을 담은 KeyedVectors 객체
# len(w2v.wv) : 벡터가 만들어진 단어 개수 (min_count 조건을 통과한 단어 수)
print("학습한 단어 수:", len(w2v.wv))


# w2v.wv["배송"]
#   - '배송'의 100칸짜리 벡터(numpy 배열)
#   - 학습하지 않은 단어를 넣으면 KeyError가 발생함
# [:5]      : 앞 5칸만 잘라냄
# .round(2) : 소수점 둘째 자리까지 반올림
print("'배송'의 앞 5칸:", w2v.wv["배송"][:5].round(2))

학습한 단어 수: 1206
'배송'의 앞 5칸: [ 0.62 -0.4   0.36  0.05  0.22]


### 가까운 단어 찾기
**✏️ [입력]** `my_words`에 궁금한 단어를 넣어 보세요. (학습한 단어만 가능)

In [5]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 입력한 단어마다 벡터가 가장 가까운 단어 5개를 출력
# ------------------------------------------------------------
# ===== ✏️ [입력] 가까운 단어를 찾을 단어 =====
my_words = ["배송", "화질", "배터리", "게임"]
# ===========================================

# 입력한 단어를 하나씩 꺼내서 처리
for w in my_words:
    # w in w2v.wv : 그 단어가 학습된 단어 목록에 있는지 검사 (True/False)
    if w in w2v.wv:

        # w2v.wv.most_similar(w, topn=5)
        #   - w의 벡터와 코사인 유사도가 가장 높은 단어 5개를
        #     (단어, 유사도) 튜플의 리스트로 유사도가 높은 순서대로 돌려줌
        #   - w 자기 자신은 결과에서 빠짐
        # [s for s, _ in ...]
        #   - 튜플에서 단어(s)만 꺼내 리스트로 만듦 ('_'는 사용하지 않는 유사도 값)
        print(w, "→", [s for s , _ in w2v.wv.most_similar(w, topn=5)])
    else:
        # 학습된 단어 목록에 없으면 안내 문구 출력
        print(w, "→ 학습하지 않은 단어입니다")

배송 → ['뽁뽁이', '예상', '예정일', '도착', '예정']
화질 → ['피로도', '또렷하다', '피로감', '뚜렷하다', '피곤']
배터리 → ['닳다', '얼마', '베터리', '짧다', '타임']
게임 → ['돌리다', '겜', '버퍼링', '플레이', '딸리다']


In [6]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 단어 쌍 3개의 코사인 유사도를 출력
# ------------------------------------------------------------

# (단어1, 단어2) 튜플 3개를 하나씩 꺼내 w1, w2에 나눠 담음
for w1, w2 in [("배송", "택배"), ("빠르다", "느리다"), ("배송", "화질")]:
    # w2v.wv.similarity(w1, w2)
    #   - 두 단어 벡터의 코사인 유사도 (범위 -1 ~ 1, 1에 가까울수록 방향이 같음)
    #   - 둘 중 하나라도 학습하지 않은 단어면 KeyError 발생
    # :.2f : 소수점 둘째 자리까지 표시하는 형식
    print(f"{w1} - {w2}: {w2v.wv.similarity(w1, w2):.2f}")

배송 - 택배: 0.71
빠르다 - 느리다: 0.61
배송 - 화질: 0.33


✅ **결과 확인**
- `배송`-`택배`는 가깝고, `배송`-`화질`은 멉니다.
- ⚠️ `빠르다`와 `느리다`는 **뜻이 반대**인데도 `배송`-`화질`보다 **가깝게** 나옵니다. 둘 다 "배송이 ___"처럼 **같은 자리**에 쓰이기 때문입니다.

> **"가깝다 = 뜻이 같다"가 아니라 "비슷한 자리에 쓰인다"** 입니다.

---
## ② 문장 임베딩: 사전학습 모델
Word2Vec은 **단어** 하나하나의 숫자 목록입니다. 문장 전체의 뜻을 담으려면 **문장 임베딩**을 씁니다.
이미 아주 많은 문장으로 학습해 둔 **사전학습 모델**을 내려받아 씁니다.


> ⚠️ 처음 실행할 때 모델을 내려받느라 1~3분 걸릴 수 있습니다.

In [7]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 사전학습 문장 임베딩 모델을 불러오고, 문장 하나를 벡터로 바꿔 보기
# ------------------------------------------------------------

# sentence_transformers 패키지에서 SentenceTransformer 클래스와 util 모듈을 불러옴
from sentence_transformers import SentenceTransformer, util
import numpy as np

# SentenceTransformer("모델 이름")
#   - Hugging Face Hub에서 해당 이름의 모델 파일(설정, 가중치, 토크나이저)을 내려받아 불러옴
#   - 한 번 내려받은 파일은 로컬 캐시에 저장되어 다음 실행부터는 캐시를 사용함
# jhgan/ko-sroberta-multitask
# 유형: 한국어 특화 SBERT (KLUE RoBERTa 기반)
# 차원: 768차원 (MRL 지원 768~32)
# 특징: KorNLI / KorSTS 기반으로 학습되어 한국어 문장 유사도 및 의미 비교에 뛰어남

st_model = SentenceTransformer("jhgan/ko-sroberta-multitask")

# st_model.encode(문자열 하나)
#   - 문장 하나를 1차원 numpy 배열(벡터)로 변환
#   - 형태소 분석을 하지 않은 원문을 그대로 넣음 (모델이 자체 토크나이저로 나눔)
vec = st_model.encode("배송이 느려요")

# len(vec): 벡터의 칸 수 (이 모델은 768)
print("숫자 개수:", len(vec))
# 벡터 앞 5칸을 소수점 둘째 자리까지 반올림해 출력
print("앞 5칸:", vec[:5].round(2))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

숫자 개수: 768
앞 5칸: [-0.57  0.4  -0.05 -0.22 -0.07]


> 🔧 **문제가 생기면:**
> 아래 명령을 새 셀에서 실행하고, 메뉴 **런타임 → 세션 다시 시작** 후 처음부터 다시 실행하세요.
> `!pip install -q -U sentence-transformers transformers`

### TF-IDF와 나란히 비교하기
같은 문장 쌍을 **TF-IDF**와 **사전학습 모델**로 각각 비교합니다.
- TF-IDF는 단원 3과 같이 **나눈 단어**로 계산합니다.
- 사전학습 모델은 **원래 문장 그대로** 넣습니다. (모델이 알아서 나눔)

In [11]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 같은 문장 쌍을 TF-IDF와 문장 임베딩으로 각각 벡터화하고
#                  두 방법의 코사인 유사도를 표로 비교
# ------------------------------------------------------------

# TfidfVectorizer: 문서를 TF-IDF 가중치 벡터로 바꾸는 scikit-learn 도구
from sklearn.feature_extraction.text import TfidfVectorizer
# cosine_similarity: 두 벡터(또는 행렬의 행들) 사이의 코사인 유사도를 계산하는 함수
from sklearn.metrics.pairwise import cosine_similarity

# TfidfVectorizer(analyzer=lambda words: words)
#   - analyzer에 함수를 넣으면, 각 문서를 그 함수에 넣어 나온 결과를 토큰 목록으로 사용
#   - 여기서는 문서가 이미 단어 리스트이므로, 받은 리스트를 그대로 돌려주는 함수를 사용
#   - 따라서 scikit-learn 자체의 토큰 분리·소문자 변환은 적용되지 않음
tfidf = TfidfVectorizer(analyzer=lambda words: words)

# tfidf.fit(df["단어"])
#   - 전체 리뷰의 단어 리스트로 단어 목록(vocabulary)과 단어별 IDF 값을 계산해 저장
#   - 벡터 변환은 하지 않음 (변환은 뒤의 transform에서 수행)
tfidf.fit(df["단어"])

# ===== ✏️ [입력] 비교할 문장 쌍 =====
# (문장1, 문장2) 튜플의 리스트, 원문 형태로 적음
pairs = [
    ("배송이 느려요", "택배가 늦게 왔어요"),      # 뜻이 비슷함, 겹치는 단어 없음
    ("배송이 느려요", "화질이 선명해요"),        # 관계없음
    ("배송이 빨라요", "배송이 느려요"),          # 뜻이 반대
]
# ===================================

# 문장 쌍별 결과(딕셔너리)를 담을 빈 리스트
rows = []

# 문장 쌍을 하나씩 꺼내 s1, s2에 나눠 담음
for s1, s2 in pairs:
    # [TF-IDF 벡터 만들기]
    #   kiwi.tokenize(s1)      : 문자열 하나를 넣으면 그 문장의 토큰 목록을 돌려줌
    #   to_words(...)          : 위에서 정의한 함수로 단어 리스트를 만듦 (리뷰를 나눌 때와 같은 규칙)
    #   tfidf.transform([...]) : 단어 리스트 2개를 TF-IDF 벡터 2개로 변환
    #     · 결과는 (2 × 단어 수) 크기의 희소 행렬, 행 0 = s1, 행 1 = s2
    #     · fit 때 단어 목록에 없던 단어는 무시됨
    t = tfidf.transform([to_words(kiwi.tokenize(s1)), to_words(kiwi.tokenize(s2))])   # TF-IDF 벡터 2개

    # [임베딩 벡터 만들기]
    #   st_model.encode([s1, s2]) : 원문 문장 2개를 (2 × 768) 크기의 numpy 배열로 변환
    #     · 행 0 = s1의 벡터, 행 1 = s2의 벡터
    e = st_model.encode([s1, s2])                                                    # 임베딩 벡터 2개

    # 한 쌍의 결과를 딕셔너리로 만들어 rows에 추가
    #   cosine_similarity(t[0], t[1])
    #     - t[0], t[1]은 각각 (1 × 단어 수) 크기의 희소 행렬
    #     - 결과는 (1 × 1) 배열이므로 [0, 0]으로 숫자 하나를 꺼냄
    #     - 두 벡터 중 하나가 모두 0이면(겹치는 단어가 없으면) 결과는 0
    #   cosine_similarity([e[0]], [e[1]])
    #     - e[0], e[1]은 1차원 배열이므로 리스트로 한 번 감싸 2차원(1 × 768) 형태로 넘김
    #     - 결과 (1 × 1) 배열에서 [0, 0]으로 숫자 하나를 꺼냄
    rows.append({"문장1": s1, "문장2": s2,
                 "TF-IDF": cosine_similarity(t[0], t[1])[0, 0],
                 "임베딩": cosine_similarity([e[0]], [e[1]])[0, 0]})

# 딕셔너리 리스트를 DataFrame으로 만들고(키 = 열 이름), 숫자를 소수점 둘째 자리까지 반올림해 출력
pd.DataFrame(rows).round(2)

,문장1,문장2,TF-IDF,임베딩
0,배송이 느려요,택배가 늦게 왔어요,0.00,0.64
1,배송이 느려요,화질이 선명해요,0.00,0.07
2,배송이 빨라요,배송이 느려요,0.36,0.56


✅ **결과 확인**
- 0·1행: TF-IDF는 겹치는 단어가 없어 둘 다 **0**입니다. 임베딩은 뜻이 비슷한 0행이 1행보다 **높게** 나오나요?
- 1행: 관계없는 문장이라 임베딩도 **낮게** 나오나요?
- 2행: ⚠️ 뜻이 반대인데 임베딩이 꽤 **높게** 나올 수 있습니다. 둘 다 "배송 속도" 이야기라서입니다.

> 💡 유사도 숫자는 **모델마다 범위가 달라서** "0.7 이상이면 비슷하다" 같은 정해진 기준이 없습니다. **순위**(어느 쪽이 더 높은가)로 보세요.

---
## ③ 임베딩 검색
질문 문장을 넣으면 **뜻이 비슷한 리뷰**를 찾아 줍니다.

### 💡 왜 리뷰를 문장으로 나눌까?
리뷰 하나에 여러 이야기가 섞여 있으면, 리뷰 전체를 숫자 목록 하나로 만들 때 찾는 내용이 **묻혀 버립니다.**

```
"화면이 크고 선명해요. 배터리는 좀 빨리 닳네요."
 → 문장 1: 화면이 크고 선명해요.
 → 문장 2: 배터리는 좀 빨리 닳네요.   ← "충전해도 금방 꺼져요"와 직접 비교됨
```

### 💡 검색 순서
1. 리뷰를 **문장**으로 나누고, 문장을 모두 숫자 목록으로 바꿔 둔다 (**색인 만들기**)
2. 질문도 같은 방법으로 숫자 목록으로 바꾼다
3. 유사도가 가장 큰 문장을 찾고, 그 문장이 들어 있는 **리뷰**를 보여 준다

> ⏱️ 리뷰 6,001건은 문장 약 1만 7천 개가 됩니다. **GPU**면 금방 끝나고, CPU면 10~15분 걸릴 수 있습니다. 현 실습에서는 `SAMPLE_N`=1000으로 합니다.

In [9]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 검색 대상 리뷰를 고르고, 리뷰를 문장 단위로 나눈 뒤
#                  문장마다 TF-IDF 벡터와 임베딩 벡터를 미리 만들어 둠(문장 단위 색인)
# ------------------------------------------------------------

# ===== ✏️ [입력] 검색에 쓸 리뷰 수 (CPU에서 오래 걸리면 1000 정도로 줄이기) =====
# 무작위로 뽑을 리뷰 개수
SAMPLE_N = 1000   #검색 성능 향상을 하려면 6000 입력(15분걸림)
# ===========================================================================

# min(SAMPLE_N, len(df))
#   - SAMPLE_N과 전체 리뷰 수 중 작은 값을 사용
#   - df.sample은 중복 없이 뽑을 때 전체 행 수보다 큰 수를 넣으면 ValueError가 발생하므로, 그 경우를 막음
# df.sample(개수, random_state=42)
#   - df에서 지정한 개수만큼 행을 중복 없이 무작위로 뽑음
#   - random_state=42 : 난수 시작값을 고정해서 실행할 때마다 같은 행들이 뽑힘
# .reset_index(drop=True)
#   - 행 번호를 0부터 다시 매기고, 원래 행 번호는 열로 남기지 않고 버림
#   - 이후 '리뷰번호'(0, 1, 2, ...)가 pool의 행 번호와 같아짐
pool = df.sample(min(SAMPLE_N, len(df)), random_state=42).reset_index(drop=True)

# 문장 하나당 딕셔너리 하나({"리뷰번호": ..., "문장": ...})를 담을 빈 리스트
rows = []

# kiwi.split_into_sents(문자열 목록)
#   - 리뷰 문자열 목록을 넘기면, 리뷰마다 '문장 객체들의 리스트'를 입력 순서대로 하나씩 돌려줌
#   - 문장 객체 s는 s.text 속성에 그 문장의 문자열을 가짐
# enumerate(...)
#   - 0부터 시작하는 순번 i를 함께 돌려줌
#   - i = pool에서의 리뷰 위치(= 리셋된 행 번호)
#   - sents = 그 리뷰를 나눈 문장 객체 리스트
for i, sents in enumerate(kiwi.split_into_sents(pool["상품평"].tolist())):
    # 한 리뷰 안의 문장을 하나씩 꺼냄
    for s in sents:
        # 원래 리뷰 번호와 문장 문자열을 짝지어 rows에 추가
        rows.append({"리뷰번호": i, "문장": s.text})

# 딕셔너리 리스트를 DataFrame으로 변환 (열: '리뷰번호', '문장' / 행: 문장 하나)
sent = pd.DataFrame(rows)

# 뽑은 리뷰 수와 나뉜 문장 수 출력
print("리뷰 수:", len(pool), "→ 문장 수:", len(sent))

# 색인 ① TF-IDF (문장을 단어로 나눠서 사용)
#   sent["문장"].tolist()        : 모든 문장을 문자열 리스트로 변환
#   kiwi.tokenize(문자열 목록)   : 문장마다 형태소 분석한 토큰 목록을 순서대로 하나씩 돌려줌
#   to_words(t)                  : 다른 셀에서 정의한 함수로 문장 하나의 단어 리스트를 만듦
#   [ ... for t in ... ]         : 모든 문장의 단어 리스트를 모은 리스트
#   tfidf.transform([...])       : 다른 셀에서 fit한 tfidf로 단어 리스트들을 TF-IDF 벡터로 변환
#     · 단어 목록과 IDF 값은 fit할 때 사용한 데이터 기준 값을 그대로 사용
#     · 결과: (문장 수 × 단어 수) 크기의 희소 행렬, 행 순서 = sent의 행 순서
X_tfidf = tfidf.transform([to_words(t) for t in kiwi.tokenize(sent["문장"].tolist())])

# 색인 ② 임베딩 (원문 문장을 그대로 사용, 진행 막대 표시)
#   sent["문장"].tolist()
#     - 모든 문장을 문자열 리스트로 변환
#   st_model.encode(리스트, show_progress_bar=True)
#     - 다른 셀에서 불러온 문장 임베딩 모델로 문자열들을 벡터로 변환
#     - show_progress_bar=True : 묶음(batch) 단위 진행 막대를 표시
#     - 결과: (문장 수 × 모델의 벡터 차원 수) 크기의 numpy 배열, 행 순서 = sent의 행 순서
X_emb = st_model.encode(sent["문장"].tolist(), show_progress_bar=True)

# .shape: (행 수, 열 수) = (문장 수, 벡터 차원 수)
print("색인 준비 완료:", X_emb.shape)

리뷰 수: 1000 → 문장 수: 2908


Batches:   0%|          | 0/91 [00:00<?, ?it/s]

색인 준비 완료: (2908, 768)


**✏️ [입력]** `query`에 질문 문장을 넣어 보세요. 리뷰에 **그대로 나오지 않을 표현**으로 쓰면 차이가 잘 보입니다.
예) `충전해도 금방 꺼져요`,

In [10]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 질문 문장과 모든 문장의 유사도를 두 방법으로 계산하고,
#                  방법별로 유사도가 높은 문장을 '리뷰 하나당 하나씩' 골라 5개를
#                  그 문장이 들어 있는 원래 리뷰와 함께 출력
# ------------------------------------------------------------

# ===== ✏️ [입력] 질문 문장 =====
# 원문 형태로 적음 (TF-IDF용 단어 나누기는 아래 코드에서 수행)
query = "충전해도 금방 꺼져요"
# =============================

# [TF-IDF 유사도]
#   kiwi.tokenize(query)               : 문자열 하나를 넣으면 그 문장의 토큰 목록을 돌려줌
#   to_words(...)                      : 다른 셀에서 정의한 함수로 질문의 단어 리스트를 만듦 (색인 문장과 같은 규칙)
#   tfidf.transform([...])             : 질문의 TF-IDF 벡터, (1 × 단어 수) 희소 행렬
#     · fit할 때 단어 목록에 없던 단어는 무시됨
#   cosine_similarity(질문 벡터, X_tfidf) : 질문과 모든 문장의 코사인 유사도, (1 × 문장 수) 배열
#   [0]                                : 첫 번째 행을 꺼내 길이 '문장 수'의 1차원 배열로 만듦
#   sent["TF-IDF"] = ...               : 문장별 유사도를 sent의 'TF-IDF' 열로 저장 (배열 순서 = sent 행 순서 = X_tfidf 행 순서)
sent["TF-IDF"] = cosine_similarity(tfidf.transform([to_words(kiwi.tokenize(query))]), X_tfidf)[0]

# [임베딩 유사도]
#   st_model.encode([query])             : 질문 원문을 리스트로 감싸 넣어 (1 × 벡터 차원 수) 배열로 변환
#   cosine_similarity(질문 벡터, X_emb)    : 질문과 모든 문장의 코사인 유사도, (1 × 문장 수) 배열
#   [0]                                  : 1차원 배열로 꺼냄
#   sent["임베딩"] = ...                  : 문장별 유사도를 sent의 '임베딩' 열로 저장
sent["임베딩"] = cosine_similarity(st_model.encode([query]), X_emb)[0]

# 두 방법을 차례대로 처리
for method in ["TF-IDF", "임베딩"]:
    # 방법 이름을 제목 줄로 출력
    print(f"===== {method} =====")

    # sent.sort_values(method, ascending=False)
    #   - 해당 방법의 유사도 열을 기준으로 큰 값부터 정렬
    # .drop_duplicates("리뷰번호")
    #   - '리뷰번호'가 같은 행이 여러 개면 처음 나온 행만 남기고 나머지를 제거 (기본값 keep="first")
    #   - 이미 유사도 내림차순으로 정렬되어 있으므로, 리뷰마다 유사도가 가장 높은 문장 하나만 남음
    # .head(5)
    #   - 위에서 5개 행 = 서로 다른 리뷰 5개에서 나온 문장 5개
    top = sent.sort_values(method, ascending=False).drop_duplicates("리뷰번호").head(5)

    # top의 행을 하나씩 꺼냄 (행 번호는 '_'로 받아 사용하지 않음, row는 그 행의 Series)
    for _, row in top.iterrows():
        # row["리뷰번호"]                : 이 문장이 나온 리뷰의 pool 행 번호 (pool은 0부터 다시 번호를 매겨 둠)
        # pool["상품평"][리뷰번호]        : 그 행 번호의 리뷰 원문
        # .replace("\n", " ")           : 원문 안의 줄바꿈 문자를 모두 띄어쓰기 한 칸으로 바꿈 (한 줄로 출력하기 위함)
        review = pool["상품평"][row["리뷰번호"]].replace("\n", " ")   # 원래 리뷰의 줄바꿈을 띄어쓰기로

        # 유사도(소수점 둘째 자리)와 찾은 문장의 앞 50글자 출력
        print(f"{row[method]:.2f} | {row['문장'][:50]}")                # 찾은 문장

        # 그 문장이 들어 있는 원래 리뷰의 앞 70글자를 들여쓰기해서 출력
        print(f"       └ 리뷰: {review[:70]}")                           # 그 문장이 들어 있는 리뷰

    # 방법 사이 빈 줄 출력
    print()

===== TF-IDF =====
0.40 | 대기시간 선택시 장시간항목이 없어서 꺼지니 너무 불편해요
       └ 리뷰: 전에 사용하던 제품보다 화면이 크니 시야가 트여 영상볼때 좋네요. 대기시간 선택시 장시간항목이 없어서 꺼지니 너무 불편해요 얼
0.29 | 충전시 충전기에서 발열감이 느껴지네요.
       └ 리뷰: 가벼워 휴대하기 편하네요 충전시 충전기에서 발열감이 느껴지네요. 정상아닌거 같네요
0.25 | 충전하는데 시간이 오래걸리네요
       └ 리뷰: 충전하는데 시간이 오래걸리네요 고속충전기가 아니여서 아쉬워요 간단 작업하고 미디어 시청하는데는 충분히 잘 작동되서 좋아요 와이
0.24 | 사용해보니 배터리 충전이 빨리 떨어져요.
       └ 리뷰: 안전하게 잘 받았구요 ㆍ11로 업데이트가 되어 있으니  따로 손볼 필요가 없네요. 사용해보니 배터리 충전이 빨리 떨어져요.
0.24 | 사은품으로 45w충전기를 주셔서 너무 좋네요
       └ 리뷰: 성능대비 저렴한 가격에 잘 구입해서 만족스럽네요 사은품으로 45w충전기를 주셔서 너무 좋네요

===== 임베딩 =====
0.83 | 배터리 충전하고 금방 방전되요.......
       └ 리뷰: 할인행사쿠폰으로 싸게 샀어요~ 다 좋은데 배터리 충전하고 금방 방전되요....... 왜그럴까요...
0.80 | 다만 배터리가 금방 닳아 없어지네요
       └ 리뷰: 제품 자체의 성능은 최고예요!! 영상보기에 적당해서 좋아요~ 다만 배터리가 금방 닳아 없어지네요
0.78 | 배터리 타임이 짧아 배터리가 금방 닳아요
       └ 리뷰: 초기 설정후 자동 업데이트후 멈췄지만 설정에 들어가 변경하니 해결했어요 배터리 타임이 짧아 배터리가 금방 닳아요
0.75 | 배터리가 체감상 금방 닳는거 같아요
       └ 리뷰: 본체에 흠이나 찍힘없네요 좋은 제품받았어요 배터리가 체감상 금방 닳는거 같아요
0.75 | 배터리가 체감상 금방 닳는거 같아요
       └ 리뷰: 배터리가 체감상 금방 

✅ **결과 확인**
- **TF-IDF**는 질문의 단어(`충전`, `꺼지다`)가 **그대로 들어간** 문장을 찾습니다. 그래서 "화면이 자주 꺼지네요"(다른 문제)나 "충전이 많이 돼서 좋네요"(관계없음)가 올라옵니다.
- **임베딩**은 "배터리가 빨리 닳아요", "배터리가 얼마 못 가요"처럼 **단어가 달라도** 뜻이 비슷한 문장을 찾기도 합니다.
- ⚠️ 임베딩도 **반대 뜻** 문장이 섞일 수 있습니다. 결과는 꼭 직접 읽어 보세요.

---
## 정리

### ✅ 오늘의 한 줄 정리
**TF-IDF는 "같은 단어"를, 임베딩은 "비슷한 뜻"을 찾는다.**

| 비교 | TF-IDF | 임베딩 (사전학습 모델) |
|---|---|---|
| 기준 | 같은 단어 | 비슷한 뜻 |
| 속도·준비 | 빠르고 가벼움 | 모델 내려받기 필요, 느림 |
| 이유 설명 | 어떤 단어가 겹쳤는지 보임 | 왜 비슷한지 설명하기 어려움 |
| 잘 맞는 곳 | 제품명·모델명처럼 정확한 단어 검색 | 표현이 다양한 질문·불만 검색 |

### 🤔 생각해 볼 점 (한계)
| 한계 | 예 | 더 나은 방법 |
|---|---|---|
| **반대 뜻도 가까움** | `빠르다` ↔ `느리다`, "배송이 빨라요" ↔ "배송이 느려요" | 감성분석과 함께 쓰기 |
| **Word2Vec은 데이터에 따라 달라짐** | 리뷰 6천 건은 작은 편이라 결과가 들쭉날쭉할 수 있음 | 더 많은 데이터, 사전학습 모델 |
| **유사도 기준이 없음** | 0.6이 높은지 낮은지 모델마다 다름 | 순위로 보고, 결과를 직접 확인 |
| **검색 속도** | 문장 수가 많으면 CPU에서 오래 걸림 | GPU 사용, `SAMPLE_N` 줄이기 |



**참고:** [gensim Word2Vec](https://radimrehurek.com/gensim/models/word2vec.html) · [sentence-transformers](https://www.sbert.net/) · [ko-sroberta-multitask](https://huggingface.co/jhgan/ko-sroberta-multitask)